# SupplyShield — Person B
## Day 1: Dataset Analysis and Baseline Preparation

### Objectives
1. Understand the delivery dataset.
2. Validate data quality.
3. Analyze trip and leg structures.
4. Study the delay ratio distribution.
5. Prepare chronological train, calibration and test datasets.
6. Establish a baseline regression model.

In [2]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)

DATA_PATH = Path("../data/processed/legs.csv")
SPLIT_PATH = Path("../data/datasplit")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nFirst five records:")
display(df.head())

Dataset shape: (26369, 14)

Column names:
['trip_uuid', 'source_center', 'destination_center', 'od_start_time', 'od_end_time', 'trip_creation_time', 'source_name', 'destination_name', 'route_type', 'actual_time_min', 'osrm_time_min', 'osrm_distance_km', 'scan_rows', 'delay_ratio']

Data types:
trip_uuid              object
source_center          object
destination_center     object
od_start_time          object
od_end_time            object
trip_creation_time     object
source_name            object
destination_name       object
route_type             object
actual_time_min       float64
osrm_time_min         float64
osrm_distance_km      float64
scan_rows               int64
delay_ratio           float64
dtype: object

First five records:


,trip_uuid,source_center,destination_center,od_start_time,od_end_time,trip_creation_time,source_name,destination_name,route_type,actual_time_min,osrm_time_min,osrm_distance_km,scan_rows,delay_ratio
0,trip-153671041653548748,IND209304AAA,IND000000ACB,2018-09-12 16:39:46.858469,2018-09-13 13:40:23.123744,2018-09-12 00:00:16.535741,Kanpur_Central_H_6 (Uttar Pradesh),Gurgaon_Bilaspur_HB (Haryana),FTL,732.0,349.0,446.5496,18,2.097421
1,trip-153671041653548748,IND462022AAA,IND209304AAA,2018-09-12 00:00:16.535741,2018-09-12 16:39:46.858469,2018-09-12 00:00:16.535741,Bhopal_Trnsport_H (Madhya Pradesh),Kanpur_Central_H_6 (Uttar Pradesh),FTL,830.0,394.0,544.8027,21,2.106599
2,trip-153671042288605164,IND561203AAB,IND562101AAA,2018-09-12 02:03:09.655591,2018-09-12 03:01:59.598855,2018-09-12 00:00:22.886430,Doddablpur_ChikaDPP_D (Karnataka),Chikblapur_ShntiSgr_D (Karnataka),Carting,47.0,26.0,28.1994,3,1.807692
3,trip-153671042288605164,IND572101AAA,IND561203AAB,2018-09-12 00:00:22.886430,2018-09-12 02:03:09.655591,2018-09-12 00:00:22.886430,Tumkur_Veersagr_I (Karnataka),Doddablpur_ChikaDPP_D (Karnataka),Carting,96.0,42.0,56.9116,6,2.285714
4,trip-153671043369099517,IND000000ACB,IND160002AAC,2018-09-14 03:40:17.106733,2018-09-14 17:34:55.442454,2018-09-12 00:00:33.691250,Gurgaon_Bilaspur_HB (Haryana),Chandigarh_Mehmdpur_H (Punjab),FTL,611.0,212.0,281.2109,12,2.882075


In [3]:
overview = pd.DataFrame({
    "Metric": [
        "Total legs",
        "Unique trips",
        "Total columns",
        "Missing values",
        "Duplicate rows"
    ],
    "Value": [
        len(df),
        df["trip_uuid"].nunique(),
        len(df.columns),
        df.isna().sum().sum(),
        df.duplicated().sum()
    ]
})

display(overview)

,Metric,Value
0,Total legs,26369
1,Unique trips,14817
2,Total columns,14
3,Missing values,147
4,Duplicate rows,0


## Feature Selection and Target Definition

The objective is to predict the delay ratio of a delivery leg using
information available before or during the prediction time.

Target variable: delay_ratio

We must exclude information that becomes available only after
the delivery leg has been completed.

In [4]:
feature_info = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Missing Values": df.isna().sum().values,
    "Unique Values": df.nunique().values
})

display(feature_info)

,Column,Data Type,Missing Values,Unique Values
0,trip_uuid,object,0,14817
1,source_center,object,0,1508
2,destination_center,object,0,1481
3,od_start_time,object,0,26369
4,od_end_time,object,0,26369
5,trip_creation_time,object,0,14817
6,source_name,object,66,1498
7,destination_name,object,81,1468
8,route_type,object,0,2
9,actual_time_min,float64,0,1658


In [5]:
# Convert timestamp columns to datetime

time_columns = [
    "trip_creation_time",
    "od_start_time",
    "od_end_time"
]

for col in time_columns:
    df[col] = pd.to_datetime(df[col])

# Test A: Trip creation vs leg start

df["create_to_start_hours"] = (
    df["od_start_time"] - df["trip_creation_time"]
).dt.total_seconds() / 3600

print("TEST A: TRIP CREATION")
print(
    "Legs starting before trip creation:",
    (df["create_to_start_hours"] < 0).sum()
)
display(df["create_to_start_hours"].describe())

# Test B: Leg start vs leg end

df["duration_calculated_min"] = (
    df["od_end_time"] - df["od_start_time"]
).dt.total_seconds() / 60

print("\nTEST B: LEG DURATION")
print(
    "Legs ending at or before start:",
    (df["duration_calculated_min"] <= 0).sum()
)

# Test C: Compare calculated duration with actual_time_min

df["duration_difference"] = (
    df["duration_calculated_min"] - df["actual_time_min"]
)

print("\nTEST C: ACTUAL DURATION COMPARISON")
display(df["duration_difference"].describe())

print(
    "Share matching within 1 minute:",
    round((df["duration_difference"].abs() < 1).mean(), 3)
)

# Test D: Scan rows and delay ratio relationship

print("\nTEST D: SCAN ROWS CORRELATION")

display(
    df[["scan_rows", "delay_ratio"]].corr(method="spearman")
)

TEST A: TRIP CREATION
Legs starting before trip creation: 0


count    26369.000000
mean         3.866803
std          5.930097
min          0.000000
25%          0.000000
50%          1.614896
75%          5.745464
max         59.583578
Name: create_to_start_hours, dtype: float64


TEST B: LEG DURATION
Legs ending at or before start: 0

TEST C: ACTUAL DURATION COMPARISON


count    26369.000000
mean        98.085567
std        136.170823
min          0.001223
25%         23.475726
50%         49.589097
75%        121.572421
max       5357.551955
Name: duration_difference, dtype: float64

Share matching within 1 minute: 0.006

TEST D: SCAN ROWS CORRELATION


,scan_rows,delay_ratio
scan_rows,1.000000,-0.060167
delay_ratio,-0.060167,1.000000


In [6]:
duration_check = df[
    [
        "trip_uuid",
        "source_center",
        "destination_center",
        "od_start_time",
        "od_end_time",
        "actual_time_min",
        "osrm_time_min"
    ]
].copy()

duration_check["timestamp_duration_min"] = (
    duration_check["od_end_time"] -
    duration_check["od_start_time"]
).dt.total_seconds() / 60

duration_check["difference_min"] = (
    duration_check["timestamp_duration_min"] -
    duration_check["actual_time_min"]
)

display(duration_check.head(5))

,trip_uuid,source_center,destination_center,od_start_time,od_end_time,actual_time_min,osrm_time_min,timestamp_duration_min,difference_min
0,trip-153671041653548748,IND209304AAA,IND000000ACB,2018-09-12 16:39:46.858469,2018-09-13 13:40:23.123744,732.0,349.0,1260.604421,528.604421
1,trip-153671041653548748,IND462022AAA,IND209304AAA,2018-09-12 00:00:16.535741,2018-09-12 16:39:46.858469,830.0,394.0,999.505379,169.505379
2,trip-153671042288605164,IND561203AAB,IND562101AAA,2018-09-12 02:03:09.655591,2018-09-12 03:01:59.598855,47.0,26.0,58.832388,11.832388
3,trip-153671042288605164,IND572101AAA,IND561203AAB,2018-09-12 00:00:22.886430,2018-09-12 02:03:09.655591,96.0,42.0,122.779486,26.779486
4,trip-153671043369099517,IND000000ACB,IND160002AAC,2018-09-14 03:40:17.106733,2018-09-14 17:34:55.442454,611.0,212.0,834.638929,223.638929


In [7]:
import numpy as np

train = pd.read_csv("../data/datasplit/train_legs_v1.csv")
cal = pd.read_csv("../data/datasplit/calibration_legs_v1.csv")
test = pd.read_csv("../data/datasplit/test_legs_v1.csv")

def evaluate_baseline(data, predictions):

    actual_ratio = data["delay_ratio"].to_numpy()

    actual_excess = (
        (data["delay_ratio"] - 1) * data["osrm_time_min"]
    ).to_numpy()

    predicted_excess = (
        (predictions - 1) * data["osrm_time_min"]
    )

    return {
        "MAE_ratio": float(
            np.abs(actual_ratio - predictions).mean()
        ),
        "MedAE_ratio": float(
            np.median(np.abs(actual_ratio - predictions))
        ),
        "MAE_excess_min": float(
            np.abs(actual_excess - predicted_excess).mean()
        )
    }


# Baseline 1: Global median
global_median = float(train["delay_ratio"].median())

# Baseline 2: Lane median, calculated using training data only
lane_medians = train.groupby(
    ["source_center", "destination_center"]
)["delay_ratio"].median()

def predict_lane(data):

    predictions = []

    for source, destination in zip(
        data["source_center"],
        data["destination_center"]
    ):
        predictions.append(
            lane_medians.get(
                (source, destination),
                global_median
            )
        )

    return np.array(predictions)


# Evaluate both baselines
results = {}

for name, data in [
    ("calibration", cal),
    ("test", test)
]:

    global_predictions = np.full(
        len(data), global_median
    )

    lane_predictions = predict_lane(data)

    results[name] = {
        "global_median": evaluate_baseline(
            data, global_predictions
        ),
        "lane_median": evaluate_baseline(
            data, lane_predictions
        )
    }

print("Training global median:", global_median)

display(
    pd.json_normalize(results, sep="_").T
)

Training global median: 2.0


,0
calibration_global_median_MAE_ratio,0.964829
calibration_global_median_MedAE_ratio,0.436019
calibration_global_median_MAE_excess_min,47.369417
calibration_lane_median_MAE_ratio,0.639492
calibration_lane_median_MedAE_ratio,0.247619
calibration_lane_median_MAE_excess_min,29.318282
test_global_median_MAE_ratio,1.015035
test_global_median_MedAE_ratio,0.448276
test_global_median_MAE_excess_min,49.306199
test_lane_median_MAE_ratio,0.740583


In [8]:
train_lanes = set(
    zip(train["source_center"], train["destination_center"])
)

for name, data in [("Calibration", cal), ("Test", test)]:
    data_lanes = list(
        zip(data["source_center"], data["destination_center"])
    )

    unseen = sum(lane not in train_lanes for lane in data_lanes)

    print(f"{name}:")
    print("Total legs:", len(data))
    print("Unseen lane legs:", unseen)
    print("Unseen lane percentage:", round(unseen / len(data) * 100, 2), "%")
    print()

Calibration:
Total legs: 5441
Unseen lane legs: 238
Unseen lane percentage: 4.37 %

Test:
Total legs: 5307
Unseen lane legs: 434
Unseen lane percentage: 8.18 %



In [9]:
from pathlib import Path
import json

output_path = Path("../experiments/day1/baseline_metrics_v1.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

if output_path.exists():
    raise FileExistsError(
        f"{output_path} already exists. No file was overwritten."
    )

baseline_summary = {
    "experiment": "Day 1 Baseline Evaluation",
    "target": "delay_ratio",
    "global_training_median": global_median,
    "prediction_strategy": {
        "known_lane": "Use training lane median",
        "unseen_lane": "Use training global median"
    },
    "dataset_sizes": {
        "training_legs": len(train),
        "calibration_legs": len(cal),
        "test_legs": len(test)
    },
    "unseen_lanes": {
        "calibration": {
            "legs": 238,
            "percentage": 4.37
        },
        "test": {
            "legs": 434,
            "percentage": 8.18
        }
    },
    "metrics": results
}

with open(output_path, "w") as f:
    json.dump(baseline_summary, f, indent=4)

print("Baseline results saved to:", output_path)

Baseline results saved to: ..\experiments\day1\baseline_metrics_v1.json


# Day 1 — Dataset Analysis & Baseline Evaluation

## 1. Dataset overview
- Total legs: 26,369
- Unique trips: 14,817
- Total features: 14
- Date range (`od_start_time`): 12 September – 6 October 2018
- No missing values in the main numerical and identifier columns.
- No exact duplicate rows.

## 2. Important observations
- Actual travel time is strongly right-skewed, with some extreme values.
- `actual_time_min` does not match the duration calculated from `od_start_time` and `od_end_time`.
- Every trip has exactly one `trip_creation_time`.
- Approximately 86.45% of consecutive leg pairs within trips are connected.
- Delay ratio is a continuous target, so this is a regression problem.

## 3. Prediction target and timing
- Prediction target: `delay_ratio = actual_time_min / osrm_time_min`
- Proposed prediction moment: immediately before or at leg departure.
- Predicted excess delay minutes: `(predicted_delay_ratio - 1) × osrm_time_min`.

## 4. Initial feature decisions
Potential pre-departure features:
- `source_center`
- `destination_center`
- `route_type`
- `osrm_time_min`
- `osrm_distance_km`
- Time-based features from `od_start_time`
- Potentially `trip_creation_time`-based features, subject to availability confirmation.

Excluded from the initial pre-departure feature set:
- `actual_time_min`, `delay_ratio`, `od_end_time`: outcome or post-completion information.
- `trip_uuid`: identifier, retained for grouping and splitting.
- `source_name`, `destination_name`: redundant with center IDs and contain missing values.
- `scan_rows`: timing and availability of scan events are not established.

## 5. Data splitting
Chronological, trip-level split using `trip_creation_time`:

| Partition | Trips | Legs |
|---|---:|---:|
| Training | 8,890 | 15,621 |
| Calibration | 2,963 | 5,441 |
| Test | 2,964 | 5,307 |

No trip overlap between partitions.

## 6. Baseline evaluation
Two baselines were evaluated:
1. Global median delay ratio.
2. Lane median delay ratio, with global median fallback for unseen lanes.

The lane median performed better on both calibration and test data.

- Training global median: 2.0
- Calibration lane median MAE: 0.6395
- Test lane median MAE: 0.7405
- Unseen lane legs: 4.37% in calibration and 8.18% in test.

Results saved in `experiments/day1/baseline_metrics_v1.json`.

## 7. Decisions and pending confirmations
- Keep the original dataset unchanged.
- Use the lane median as the stronger baseline.
- Confirm with Person A whether planned route features and timestamped scan events are available at prediction time.
- Confirm target definition and units with Person C.

## 8. Next steps
- Build the first XGBoost regression model.
- Evaluate on calibration and test data.
- Develop Conformal Prediction intervals.
- Compare model performance against the baseline.

## Future improvements / ideas to revisit
- Explore dynamic delay prediction using timestamped scan events accumulated up to the prediction time. This requires scan-event timestamps and careful prevention of future-information leakage.

In [11]:
df["route_type"].value_counts(dropna=False)

route_type
FTL        13940
Carting    12429
Name: count, dtype: int64